In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import os
from scipy.io import loadmat, savemat
import warnings
import time


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.fc4(x)
        return x

In [ ]:
# Training loop
def train_model():

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0
        model.train()
        permutation = torch.randperm(X_train_tensor.size(0))
        for i in range(0, X_train_tensor.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]

            # Forward pass
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            epoch_train_loss += loss.item()

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_train_loss /= len(X_train_tensor) // batch_size
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_train_loss:.4f}')

In [ ]:
# Evaluate the model
def evaluate_model():
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predicted = torch.max(outputs, 1)
        # Convert to CPU for compatibility with sklearn metrics
        predicted_np = predicted.cpu().numpy()
        y_test_np = y_test_tensor.cpu().numpy()

        # Calculate accuracy and recall
        accuracy = accuracy_score(y_test_np, predicted_np)
        recall = recall_score(y_test_np, predicted_np)

        print(f'MLP Test Accuracy: {accuracy * 100:.2f}%')
        print(f'MLP Test Recall: {recall * 100:.2f}%')

        # Save predicted data for current exclude_idx
        save_directory = f"/content/drive/MyDrive/Mesh_segmentation_64/Training_testing/nonweighted/Predicted_Cell_{cell_idx}"
        os.makedirs(save_directory, exist_ok=True)
        save_path = os.path.join(save_directory, f"predicted_exclude_idx_{exclude_idx}.mat")

        savemat(save_path, {"predicted": predicted_np, "y_test": y_test})
        print(f"Saved predicted data for cell_idx {cell_idx}, exclude_idx {exclude_idx} to {save_path}")

In [ ]:
# Training loop
def train_model_with_class_weight():

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0
        model_class_weight.train()
        permutation = torch.randperm(X_train_tensor.size(0))
        for i in range(0, X_train_tensor.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]

            # Forward pass
            outputs = model_class_weight(batch_X)
            loss = criterion_class_weight(outputs, batch_y)
            epoch_train_loss += loss.item()

            # Backward pass and optimization
            optimizer_class_weight.zero_grad()
            loss.backward()
            optimizer_class_weight.step()

        epoch_train_loss /= len(X_train_tensor) // batch_size
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_train_loss:.4f}')

In [ ]:
# Evaluate the model
def evaluate_model_with_class_weight():
    model.eval()
    with torch.no_grad():
        outputs = model_class_weight(X_test_tensor)
        _, predicted = torch.max(outputs, 1)
        # Convert to CPU for compatibility with sklearn metrics
        predicted_np = predicted.cpu().numpy()
        y_test_np = y_test_tensor.cpu().numpy()

        # Calculate accuracy and recall
        accuracy = accuracy_score(y_test_np, predicted_np)
        recall = recall_score(y_test_np, predicted_np)

        print(f'MLP Test Accuracy: {accuracy * 100:.2f}%')
        print(f'MLP Test Recall: {recall * 100:.2f}%')
        # Define the specific directory and filename
        # Save predicted data for current exclude_idx
        save_directory = f"/content/drive/MyDrive/Mesh_segmentation_64/Training_testing/weighted/Predicted_Cell_{cell_idx}"
        os.makedirs(save_directory, exist_ok=True)
        save_path = os.path.join(save_directory, f"predicted_exclude_idx_{exclude_idx}.mat")

        savemat(save_path, {"predicted": predicted_np, "y_test": y_test})
        print(f"Saved predicted data for cell_idx {cell_idx}, exclude_idx {exclude_idx} to {save_path}")

In [ ]:
# Initialize variables
file_range = range(3, 24)  # Represents HC003 to HC023
num_subjects = len(file_range)
#cell_idx = 10

In [ ]:
main_folder_path = r'/content/drive/MyDrive/EEG-fMRI_data_64_electrodes/SavedFeatures_64electrodes'
main_folder_label = r'/content/drive/MyDrive/EEG-fMRI_data_64_electrodes/Meshgt_64elect'
# Loop through cell_idx from 1 to 20
for cell_idx in range(1, 21):
    # Loop through files and concatenate data
    for exclude_idx in range(20,21):  # Adjust the range as needed
        exclude_subject = file_range[exclude_idx]  # Current excluded subject

        file_name_exclude = os.path.join(main_folder_path, f'HC{exclude_subject:03d}_features_vertices.mat')
        label_file_test = os.path.join(main_folder_label, f'HC{exclude_subject:03d}_mesh_gt.mat')

        # Check if files exist
        if not os.path.isfile(file_name_exclude):
            warnings.warn(f"File {file_name_exclude} does not exist. Skipping...")
            continue
        if not os.path.isfile(label_file_test):
            warnings.warn(f"File {label_file_test} does not exist. Skipping...")
            continue

        # Load data
        data_test = loadmat(file_name_exclude)
        label_test = loadmat(label_file_test)

        if 'features_vertices' not in data_test or 'Label_vertices' not in label_test:
            warnings.warn(f"Required fields missing in {file_name_exclude} or {label_file_test}. Skipping...")
            continue

        TP_index_test = label_test['Label_vertices'][cell_idx - 1][0]
        labels_test = np.zeros(data_test['features_vertices'][cell_idx - 1][0].shape[0], dtype=int)
        labels_test[TP_index_test - 1] = 1  # MATLAB to Python indexing adjustment

        temp_features_test = data_test['features_vertices'][cell_idx - 1][0]
        temp_labels_test = labels_test

        print("Done!")

        # Initialize concatenated data

        temp_features = []
        temp_label = []

        start_time = time.time()

        # Loop through files to concatenate data
        for include_idx in range(num_subjects):
            include_subject = file_range[include_idx]

            if include_subject == exclude_subject:
                continue  # Skip the excluded subject

            # Construct the file names
            file_name = os.path.join(main_folder_path, f'HC{include_subject:03d}_features_vertices.mat')
            file_name_label = os.path.join(main_folder_label, f'HC{include_subject:03d}_mesh_gt.mat')

            # Check if the files exist
            if not os.path.isfile(file_name):
                warnings.warn(f'File {file_name} does not exist. Skipping...')
                continue

            if not os.path.isfile(file_name_label):
                warnings.warn(f'File label {file_name_label} does not exist. Skipping...')
                continue

            # Load the files
            data = loadmat(file_name)
            label = loadmat(file_name_label)

            # Check if 'features_vertices' and 'Label_vertices' exist
            if 'features_vertices' not in data:
                warnings.warn(f"File {file_name} does not contain 'features_vertices'. Skipping...")
                continue

            if 'Label_vertices' not in label:
                warnings.warn(f"File {file_name_label} does not contain 'Label_vertices'. Skipping...")
                continue

            # Get the true positive index and create labels
            TP_index = label['Label_vertices'][cell_idx-1][0]
            labels = np.zeros(data['features_vertices'][cell_idx-1][0].shape[0], dtype=int)  # Initialize labels as 0 (FP)
            labels[TP_index - 1] = 1  # Adjust to 0-based indexing for Python

            temp_label.append(labels)
            temp_features.append(data['features_vertices'][cell_idx - 1][0])
      # Concatenate all features and labels
        temp_features = np.vstack(temp_features)
        temp_label = np.concatenate(temp_label)

        print(f'Label excluding HC{exclude_subject:03d} saved')

        elapsed_time = time.time() - start_time
        print(f"Time taken: {elapsed_time:.2f} seconds")
        # Load your data (replace with your actual data loading process)
        X_train = temp_features  # Replace with your actual training features
        y_train = temp_label  # Replace with your actual training labels
        X_test = temp_features_test  # Replace with your actual testing features
        y_test = temp_labels_test  # Replace with your actual testing labels
        # Standardize the data
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        # Convert data to PyTorch tensors
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.long)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.long)
        # Use GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        X_train_tensor = X_train_tensor.to(device)
        y_train_tensor = y_train_tensor.to(device)
        X_test_tensor = X_test_tensor.to(device)
        y_test_tensor = y_test_tensor.to(device)
        # Hyperparameters
        input_size = X_train.shape[1]
        hidden_size = 512  # Adjust as needed# 1024
        num_classes = 2
        learning_rate = 0.001
        num_epochs = 100
        batch_size = 1024
        patience = 5  # Early stopping patience
        # Initialize the model, loss function, and optimizer
        model = MLP(input_size, hidden_size, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # L2 regularization with weight_decay (, weight_decay=1e-4)
        # Train and evaluate the MLP model
        train_model()
        evaluate_model()

        class_sample_count_train = torch.tensor([(y_train_tensor == t).sum() for t in torch.unique(y_train_tensor, sorted=True)])
        class_weights_train = 1.0 / class_sample_count_train.float()
        class_weights_train /= class_weights_train.sum()  # Normalize weights
        class_weights_train = class_weights_train.to(device)

        # Initialize the model, loss function, and optimizer
        model_class_weight = MLP(input_size, hidden_size, num_classes).to(device)
        criterion_class_weight = nn.CrossEntropyLoss(weight=class_weights_train)
        optimizer_class_weight = optim.Adam(model_class_weight.parameters(), lr=learning_rate)  # L2 regularization with weight_decay (, weight_decay=1e-4)

        train_model_with_class_weight()
        evaluate_model_with_class_weight()


In [ ]:
#class MLP(nn.Module):
#    def __init__(self, input_size, hidden_size, num_classes):
#        super(MLP, self).__init__()
#        self.fc1 = nn.Linear(input_size, hidden_size)
#        self.relu = nn.ReLU()
#        self.dropout = nn.Dropout(p=0.2)  # 20% dropout rate
#        self.fc2 = nn.Linear(hidden_size, hidden_size)
#        self.fc3 = nn.Linear(hidden_size, num_classes)

#    def forward(self, x):
#        x = self.fc1(x)
#        x = self.relu(x)
#        x = self.dropout(x)  # Apply dropout
#        x = self.fc2(x)
#        x = self.relu(x)
#        x = self.dropout(x)  # Apply dropout
#        x = self.fc2(x)
#        x = self.relu(x)
#        x = self.dropout(x)  # Apply dropout
#        x = self.fc3(x)
#        return x